# Task 1.1 Baseline: Base Model on GSM8K

**QUESTION 1:** Run the base Qwen2.5-1.5B-Instruct model on 100 GSM8K test questions and report the accuracy. (Expect approximately 35–40%.)

In [1]:
import re
import torch
from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# System prompt: elicit step-by-step reasoning and a consistent final-answer marker.
SYSTEM_PROMPT = """You are a helpful assistant that solves grade-school math problems. Show your reasoning step by step. At the end, give your final numerical answer on a line that starts with "Answer:" followed by a single number (e.g., Answer: 42)."""

In [2]:
# ── Answer extraction ──

def extract_boxed(text):
    """Extract content from the last \\boxed{...}, handling nested braces."""
    if not text or "\\boxed{" not in text:
        return None
    start = text.rfind("\\boxed{")
    if start == -1:
        return None
    start += len("\\boxed{")
    depth = 1
    i = start
    while i < len(text) and depth > 0:
        if text[i] == "{":
            depth += 1
        elif text[i] == "}":
            depth -= 1
        i += 1
    if depth == 0:
        return text[start : i - 1].strip()
    return None


def extract_ground_truth(raw_answer, gt_format="hashmarks"):
    """Extract the ground-truth answer string from the dataset's answer field."""
    if gt_format == "hashmarks":
        m = re.search(r"####\s*(.+)", raw_answer)
        if m:
            return normalize_answer(m.group(1).strip())
        return normalize_answer(raw_answer.strip())
    if gt_format == "boxed":
        boxed = extract_boxed(raw_answer)
        if boxed is not None:
            return normalize_answer(boxed.strip())
        return normalize_answer(raw_answer.strip())
    return normalize_answer(raw_answer.strip())


def normalize_answer(s):
    """Strip and remove commas for numeric comparison."""
    return s.strip().replace(",", "").strip()


def answers_match(pred, gt):
    """Compare predicted and ground-truth answers (both normalized). GSM8K answers are integers."""
    if pred is None or gt is None:
        return False
    pred, gt = normalize_answer(pred), normalize_answer(gt)
    try:
        return int(float(pred)) == int(float(gt))
    except ValueError:
        return pred == gt


def extract_model_answer(text):
    """Extract model's final answer from generated text. Tries 'Answer:' then \\boxed{}."""
    if not text:
        return None
    # Try "Answer: 123" or "Answer: 123."
    m = re.search(r"Answer:\s*([^\s\n]+)", text, re.IGNORECASE)
    if m:
        return normalize_answer(m.group(1))
    boxed = extract_boxed(text)
    if boxed is not None:
        return normalize_answer(boxed)
    return None

In [3]:
# Load GSM8K test set and take first 100 questions (fixed subset for all experiments)
gsm8k = load_dataset("openai/gsm8k", "main", split="test")
num_test = 100
test_questions = [gsm8k[i]["question"] for i in range(num_test)]
test_ground_truth = [extract_ground_truth(gsm8k[i]["answer"]) for i in range(num_test)]
print(f"Loaded {num_test} test questions. Example GT: {test_ground_truth[0]!r}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Loaded 100 test questions. Example GT: '18'


In [4]:
# ── Model loading and batched generation ──

def load_model(model_name=MODEL_NAME):
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        attn_implementation="eager",
    )
    model.eval()
    print(f"Loaded: {model_name}")
    return model, tokenizer


def build_prompts(tokenizer, questions):
    """Build chat-formatted prompt strings for a list of questions."""
    prompts = []
    for q in questions:
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": q},
        ]
        prompts.append(
            tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
        )
    return prompts


def generate_batch(model, tokenizer, questions, max_new_tokens=1024):
    """Generate responses for a batch of questions."""
    prompts = build_prompts(tokenizer, questions)
    inputs = tokenizer(
        prompts, return_tensors="pt", padding=True, truncation=True, max_length=2048
    ).to(model.device)
    prompt_len = inputs["input_ids"].shape[1]
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    responses = []
    for i in range(len(questions)):
        new_tokens = out[i][prompt_len:]
        responses.append(tokenizer.decode(new_tokens, skip_special_tokens=True))
    return responses

In [5]:
def evaluate_gsm8k(model, tokenizer, questions, ground_truths, batch_size=16):
    """Zero-shot eval on GSM8K. Returns accuracy and list of (pred, gt, correct, raw_response)."""
    correct = 0
    records = []
    num_samples = len(questions)
    for start in tqdm(range(0, num_samples, batch_size), desc="Evaluating"):
        end = min(start + batch_size, num_samples)
        batch_q = questions[start:end]
        batch_gt = ground_truths[start:end]
        responses = generate_batch(model, tokenizer, batch_q)
        for i, (resp, gt) in enumerate(zip(responses, batch_gt)):
            pred = extract_model_answer(resp)
            is_correct = answers_match(pred, gt)
            if is_correct:
                correct += 1
            records.append({"pred": pred, "gt": gt, "correct": is_correct, "response": resp})
    accuracy = correct / num_samples
    return accuracy, records

In [6]:
# Run baseline evaluation (QUESTION 1)
model, tokenizer = load_model()
accuracy, records = evaluate_gsm8k(
    model, tokenizer, test_questions, test_ground_truth, batch_size=16
)
print(f"\nBase model accuracy on 100 GSM8K test questions: {accuracy:.2%}")
print(f"Correct: {sum(r['correct'] for r in records)} / {len(records)}")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-1.5B-Instruct


Evaluating: 100%|██████████| 7/7 [00:38<00:00,  5.50s/it]


Base model accuracy on 100 GSM8K test questions: 32.00%
Correct: 32 / 100
